# EEG versus noise — Week 1 Colab notebook

**This week's goal:** implement the BART EEG/noise experiment from A.1, document the evaluation, and save results you can discuss with Lexin.

**Upload this file to [Google Colab](https://colab.research.google.com/). Choose Runtime → Change runtime type → GPU, then run the cells in order.** Keep the `pilot` setting for your first run. No scores are prefilled.

EEG records electrical activity through sensors on the scalp. This notebook uses processed EEG features paired with sentences that participants read. BART is a pretrained language model that generates text; the added EEG encoder learns to translate those EEG features into inputs BART can use.

The default uses real EEG from one participant in two ZuCo 1.0 reading tasks, at most 256 training / 32 development / 64 test examples, and two training epochs per model. An epoch is one pass through the training examples. The development set chooses the saved model; the test set measures performance afterward. **This small pilot checks the pipeline; it is not a full reproduction of the paper's results.** The two tasks are parts of one dataset.

Downloads: approximately 1.51 GB of MATLAB data plus BART-large weights. Compact EEG features are cached. Resumable best checkpoints for the two models can take roughly 7 GB, and temporary checkpoint replacement needs extra space. Set `SAVE_RESUME_STATE=False` to save weights only at roughly half that storage. Actual memory, disk use, and training time depend on the runtime. GPU availability is not guaranteed.

Outputs include metrics, generated examples, EEG/noise plots, split and preprocessing audits, a paired uncertainty analysis, a report with a slide outline, and a downloadable results ZIP. Large checkpoints stay in your experiment folder.

Sources: [A.1 paper](https://www.nature.com/articles/s41598-025-29587-x), [official A.1 code](https://github.com/khu-aims/EEG-To-Text), [original ZuCo conversion code](https://github.com/MikeWangWZHL/EEG-To-Text), [ZuCo data](https://osf.io/q3zws/).

## 1. Install packages
Run this first in a fresh runtime. Keep Colab's supplied PyTorch/CUDA installation. If you previously imported another Transformers version, restart the session after installation.

In [ ]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.45.2', 'tokenizers==0.20.3', 'nltk==3.9.1', 'rouge==1.0.1',
    'jiwer==3.0.4', 'sacrebleu==2.4.3', 'fuzzy-match==0.0.1', 'sentencepiece',
    'scipy', 'pandas', 'matplotlib', 'tqdm', 'requests'], check=True)
print('Installation complete.')

## 2. Settings and saving
Keep `PRESET='pilot'` initially. `full_v1` expands to all available participants in SR + NR + TSR and 30 epochs; it is much more expensive and still covers only the BART portion of A.1.

`FREEZE_LM=False` updates the language model, following the one-stage training script. Setting it to True is a separate experiment. `SAVE_RESUME_STATE=True` resumes interrupted training from the **best validation epoch**, recomputing later epochs if necessary. With False, interrupted training restarts from initialization; completed models can still be evaluated again.

The paper reports seed 42, while the public script uses 312. The evaluator uses sampling alongside beam search. Both are explicit below; changing these settings creates a different run. Gradient accumulation, modern libraries, and mixed precision are additional adaptations.

Google Drive is recommended so results survive runtime termination. Mounting Drive is performed by Colab when you run the cell.

In [ ]:
from pathlib import Path
PRESET = 'pilot'                  # 'pilot' or 'full_v1'
SAVE_TO_DRIVE = True
SAVE_RESUME_STATE = True
FREEZE_LM = False
MICRO_BATCH_SIZE = 1
SEED = 42
DO_SAMPLE = True                  # Public evaluator: True; pure beam search: False.
SPLIT_POLICY = 'paper_index'      # Alternative: 'global_sentence_hash' (different protocol).
assert PRESET in {'pilot', 'full_v1'}
CFG = dict(run_name='week1_bart', preset=PRESET,
    tasks=['task1-SR','task2-NR'] if PRESET=='pilot' else ['task1-SR','task2-NR','task3-TSR'],
    subjects=['ZAB'] if PRESET=='pilot' else 'ALL',
    epochs=2 if PRESET=='pilot' else 30,
    limits={'train':256,'dev':32,'test':64} if PRESET=='pilot' else {'train':None,'dev':None,'test':None},
    seed=SEED, noise_seed=73519, generation_seed=90210, freeze_lm=FREEZE_LM,
    micro_batch=MICRO_BATCH_SIZE, effective_batch=32, lr=2e-5,
    amp=True, gradient_checkpointing=True, save_resume_state=SAVE_RESUME_STATE,
    do_sample=DO_SAMPLE, split_policy=SPLIT_POLICY, bootstrap_repeats=1000)
WORK = Path('/content/eeg_week1')
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PERSIST = Path('/content/drive/MyDrive/EEG_Week1')
else:
    PERSIST = WORK / 'saved'
    print('Local storage only: download results before the runtime ends.')
print(CFG)

## 3. Load the helper functions
Run the following cell once. It defines the data, training, and evaluation functions; it does not start training. The complete implementation is included in this notebook, with no separate helper file to upload.

In [ ]:
#@title Define data, training, evaluation, and reporting functions
import os, sys, re, json, time, math, gc, hashlib, random, contextlib, subprocess, shutil, warnings
from pathlib import Path
from collections import Counter
import importlib.util
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from scipy.io import loadmat
from tqdm.auto import tqdm
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from transformers import BartTokenizer, BartForConditionalGeneration
from nltk.translate.bleu_score import corpus_bleu, modified_precision, brevity_penalty
from rouge import Rouge
import jiwer, sacrebleu

A1_COMMIT = '00a50d002d24ef999609fb3e9e344a8b3ba42066'
MODEL_ID = 'facebook/bart-large'
MODEL_REVISION = 'cb48c1365bd826bd521f650dc2e0940aee54720c'
BANDS = ['_t1','_t2','_a1','_a2','_b1','_b2','_g1','_g2']
LENGTH = 56
CACHE_VERSION = 'a1-gd840-56-v2'
OSF_FOLDERS = {'task1-SR':'5b4eee366d4eb300106ec69c',
               'task2-NR':'5b4eee6d33f0b9000cba7151',
               'task3-TSR':'5b4eee966d4eb300106ec71c'}
HTTP = requests.Session()
HTTP.mount('https://', HTTPAdapter(max_retries=Retry(total=4, backoff_factor=1,
                           status_forcelist=[429,500,502,503,504])))

def stable_seed(key, salt):
    return int(hashlib.sha256(f'{salt}|{key}'.encode()).hexdigest()[:16],16) % (2**31-1)

def seed_all(seed):
    random.seed(seed); np.random.seed(seed % (2**32-1)); torch.manual_seed(seed)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True

def sentence_key(text):
    return hashlib.sha256(' '.join(text.casefold().split()).encode()).hexdigest()

def atomic_json(path, value):
    path = Path(path); temp = Path(str(path)+'.tmp')
    temp.write_text(json.dumps(value, indent=2, ensure_ascii=False))
    os.replace(temp,path)

def file_hash(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda:f.read(4*1024**2),b''): h.update(block)
    return h.hexdigest()

def osf_files(task):
    url=f'https://api.osf.io/v2/nodes/q3zws/files/osfstorage/{OSF_FOLDERS[task]}/?page[size]=100'
    found=[]
    while url:
        response=HTTP.get(url,timeout=60); response.raise_for_status()
        payload=response.json()
        for item in payload['data']:
            a=item['attributes']; match=re.fullmatch(r'results([A-Z]+)_(SR|NR|TSR)\.mat',a['name'])
            if match:
                found.append(dict(task=task,subject=match.group(1),name=a['name'],bytes=int(a['size']),
                    url=item['links']['download'],sha256=a.get('extra',{}).get('hashes',{}).get('sha256')))
        url=payload.get('links',{}).get('next')
    return sorted(found,key=lambda x:x['subject'])

def download_checked(info,path):
    path=Path(path); path.parent.mkdir(parents=True,exist_ok=True)
    if path.exists():
        if path.stat().st_size==info['bytes'] and (not info['sha256'] or file_hash(path)==info['sha256']):
            return path
        raise RuntimeError(f'Existing file failed verification: {path}. Remove it and retry.')
    part=Path(str(path)+'.part'); offset=part.stat().st_size if part.exists() else 0
    if offset>info['bytes']: raise RuntimeError(f'Oversized partial download: {part}. Remove it.')
    if offset<info['bytes']:
        headers={'Range':f'bytes={offset}-'} if offset else {}
        with HTTP.get(info['url'],headers=headers,stream=True,timeout=(30,180)) as response:
            response.raise_for_status()
            if response.status_code==206:
                if not response.headers.get('Content-Range','').startswith(f'bytes {offset}-'):
                    raise RuntimeError('Unexpected byte range. Partial file was not changed.')
            else: offset=0
            with part.open('ab' if offset else 'wb') as f, tqdm(total=info['bytes'],initial=offset,
                    unit='B',unit_scale=True,desc=info['name']) as bar:
                for block in response.iter_content(4*1024**2):
                    if block: f.write(block); bar.update(len(block))
    if part.stat().st_size!=info['bytes']: raise RuntimeError('Incomplete download; rerun to resume.')
    if info['sha256'] and file_hash(part)!=info['sha256']:
        raise RuntimeError(f'Hash mismatch: remove {part} and retry.')
    part.replace(path)
    return path

def write_cache(path,rows,audit,source):
    if not rows: raise RuntimeError('No valid EEG rows survived filtering.')
    meta=[{k:v for k,v in r.items() if k not in {'eeg','labels'}} for r in rows]
    temp=Path(str(path)+'.tmp')
    with temp.open('wb') as f:
        np.savez_compressed(f, features=np.concatenate([r['eeg'] for r in rows]),
            offsets=np.cumsum([0]+[len(r['eeg']) for r in rows]),labels=np.stack([r['labels'] for r in rows]),
            metadata=np.asarray(json.dumps(meta)),audit=np.asarray(json.dumps(audit)),
            source=np.asarray(json.dumps(source)),version=np.asarray(CACHE_VERSION))
    os.replace(temp,path)

def read_cache(path):
    with np.load(path,allow_pickle=False) as z:
        assert str(z['version'])==CACHE_VERSION
        features,offsets,labels=z['features'],z['offsets'],z['labels']
        rows=json.loads(str(z['metadata']))
        for i,r in enumerate(rows):
            r['eeg']=features[offsets[i]:offsets[i+1]]; r['labels']=labels[i]
        return rows,json.loads(str(z['audit'])),json.loads(str(z['source']))

class PairedDataset(Dataset):
    def __init__(self,rows,condition,pad_id,noise_seed):
        assert condition in {'EEG','noise'}
        self.rows,self.condition,self.pad_id,self.noise_seed=rows,condition,pad_id,noise_seed
    def __len__(self): return len(self.rows)
    def __getitem__(self,i):
        row=self.rows[i]; n=len(row['eeg'])
        if self.condition=='EEG':
            x=torch.zeros(LENGTH,840); x[:n]=torch.from_numpy(row['eeg'])
        else:
            g=torch.Generator().manual_seed(stable_seed(row['id'],self.noise_seed))
            x=torch.randn(LENGTH,840,generator=g)
        labels=torch.tensor(row['labels'],dtype=torch.long)
        labels[labels==self.pad_id]=-100
        return dict(x=x,mask=torch.arange(LENGTH)<n,labels=labels)

def score_predictions(rows):
    refs=[r['reference'] for r in rows]; preds=[r['prediction'] for r in rows]
    assert refs and all(r.strip() for r in refs)
    out=dict(n_examples=len(rows),empty_predictions=sum(not p.strip() for p in preds))
    with warnings.catch_warnings():
        warnings.simplefilter('ignore',UserWarning)
        for n in range(1,5):
            out[f'bleu{n}_bart_pct']=100*corpus_bleu([[r['reference_tokens']] for r in rows],
                [r['prediction_tokens'] for r in rows],weights=(1/n,)*n)
    rouge=Rouge(); scores=[]
    for pred,ref in zip(preds,refs):
        try: s=rouge.get_scores(pred,ref)[0] if pred.strip() else None
        except ValueError: s=None
        scores.append(s)
    for variant,label in [('rouge-1','rouge1'),('rouge-2','rouge2'),('rouge-l','rougeL')]:
        for field in ['p','r','f']:
            out[f'{label}_{field}_pct']=100*float(np.mean([s[variant][field] if s else 0 for s in scores]))
            nonempty=[s[variant][field] for s in scores if s]
            out[f'{label}_{field}_ignore_empty_pct']=100*float(np.mean(nonempty)) if nonempty else None
    out['wer_pct']=100*jiwer.wer(refs,preds); out['cer_pct']=100*jiwer.cer(refs,preds)
    metric=sacrebleu.metrics.BLEU(); out['sacrebleu_pct']=metric.corpus_score(preds,[refs]).score
    out['sacrebleu_signature']=str(metric.get_signature())
    return out

def paired_bootstrap(eeg_rows,noise_rows,repeats,seed):
    a={r['id']:r for r in eeg_rows}; b={r['id']:r for r in noise_rows}
    assert a.keys()==b.keys()
    groups=sorted({r['sentence_key'] for r in a.values()}); ix={x:i for i,x in enumerate(groups)}
    if len(groups)<2: raise ValueError('At least two sentence groups are needed.')
    totals=np.zeros((len(groups),2,6))
    for sample_id in a:
        assert a[sample_id]['reference']==b[sample_id]['reference']
        for j,row in enumerate([a[sample_id],b[sample_id]]):
            hyp,ref=row['prediction_tokens'],row['reference_tokens']
            p=modified_precision([ref],hyp,1); e=jiwer.process_words(row['reference'],row['prediction'])
            totals[ix[row['sentence_key']],j]+=np.array([p.numerator,p.denominator,len(hyp),len(ref),
                e.substitutions+e.deletions+e.insertions,e.hits+e.substitutions+e.deletions])
    def bleu(v):
        return 0.0 if v[2]==0 or v[0]==0 else 100*brevity_penalty(v[3],v[2])*v[0]/v[1]
    def difference(t): return np.array([bleu(t[0])-bleu(t[1]),100*(t[1,4]/t[1,5]-t[0,4]/t[0,5])])
    observed=difference(totals.sum(0)); rng=np.random.default_rng(seed)
    draws=np.stack([difference(totals[rng.integers(0,len(groups),len(groups))].sum(0)) for _ in range(repeats)])
    lo,hi=np.percentile(draws,[2.5,97.5],axis=0)
    return pd.DataFrame(dict(comparison=['BLEU1 EEG minus noise','WER noise minus EEG'],
        benefit_percentage_points=observed,bootstrap_95_low=lo,bootstrap_95_high=hi,
        sentence_groups=len(groups),bootstrap_repeats=repeats))

class EEGExperiment:
    def __init__(self,config,work,persist):
        self.cfg=dict(config); self.work=Path(work); self.persist=Path(persist)
        self.cache=self.persist/'feature_cache'
        for p in [self.work,self.persist,self.cache]: p.mkdir(parents=True,exist_ok=True)
        self.device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        assert self.cfg['effective_batch'] % self.cfg['micro_batch']==0
        self.accum=self.cfg['effective_batch']//self.cfg['micro_batch']
        torch.set_num_threads(min(4,os.cpu_count() or 1))
        os.environ['TOKENIZERS_PARALLELISM']='false'

    def setup(self):
        repo=self.work/'authors'
        if not (repo/'.git').exists():
            subprocess.run(['git','clone','--quiet','https://github.com/khu-aims/EEG-To-Text.git',str(repo)],check=True)
        subprocess.run(['git','-C',str(repo),'checkout','--quiet',A1_COMMIT],check=True)
        assert subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()==A1_COMMIT
        source=(repo/'data.py').read_text(); debug='print("EEG", input_sample[\'input_embeddings\'])'
        assert source.count(debug)==1
        quiet=self.work/'author_data_quiet.py'
        quiet.write_text(source.replace(debug,'pass  # suppress verbose tensor output'))
        def module(name,path):
            spec=importlib.util.spec_from_file_location(name,path); m=importlib.util.module_from_spec(spec)
            spec.loader.exec_module(m); return m
        self.author_data=module('a1_data',quiet)
        self.author_models=module('a1_models',repo/'model_decoding.py')
        self.tokenizer=BartTokenizer.from_pretrained(MODEL_ID,revision=MODEL_REVISION)
        print('Author commit:',A1_COMMIT,'| device:',self.device)
        if self.device.type=='cuda':
            print(torch.cuda.get_device_name(0),'| GPU GiB:',round(torch.cuda.get_device_properties(0).total_memory/2**30,1))
        else: print('CPU can prepare data. Choose a GPU before training BART-large.')

    def convert_mat(self,path,task,subject):
        contents=loadmat(path,squeeze_me=True,struct_as_record=False,variable_names=['sentenceData'])
        sentences=np.atleast_1d(contents['sentenceData']); rows=[]; rejected=Counter()
        n_total=len(sentences); train_end=int(.8*n_total); dev_end=train_end+int(.1*n_total)
        for i,sentence in enumerate(tqdm(sentences,desc=f'Extract {task}/{subject}')):
            words=getattr(sentence,'word',None)
            if words is None or isinstance(words,(float,np.floating)):
                rejected['missing_sentence_words']+=1; continue
            obj=dict(content=str(sentence.content),word=[],sentence_level_EEG={
                'mean'+b:np.asarray(getattr(sentence,'mean'+b)).reshape(-1) for b in BANDS})
            for word in np.atleast_1d(words):
                if not hasattr(word,'nFixations') or not np.asarray(word.nFixations).size: continue
                if float(np.asarray(word.nFixations).reshape(-1)[0])>0:
                    obj['word'].append(dict(content=str(word.content),word_level_EEG={'GD':{
                        'GD'+b:np.asarray(getattr(word,'GD'+b)).reshape(-1) for b in BANDS}}))
            if not 0<len(obj['word'])<=LENGTH:
                rejected['zero_or_overlength_input']+=1; continue
            sample=self.author_data.get_input_sample(obj,self.tokenizer,eeg_type='GD',bands=BANDS,
                        max_len=LENGTH,add_CLS_token=False,test_input='EEG')
            if sample is None:
                rejected['author_nan_or_shape_filter']+=1; continue
            if not torch.isfinite(sample['input_embeddings']).all():
                rejected['nonfinite_features']+=1; continue
            labels=sample['target_ids'].numpy().astype(np.int64,copy=True)
            phase='train' if i<train_end else 'dev' if i<dev_end else 'test'
            rows.append(dict(id=f'{task}/{subject}/{i:04d}',task=task,subject=subject,index=i,phase=phase,
                raw_text=obj['content'],reference=self.tokenizer.decode(labels,skip_special_tokens=True),
                sentence_key=sentence_key(obj['content']),target_truncated=len(self.tokenizer.encode(obj['content']))>LENGTH,
                labels=labels,eeg=sample['input_embeddings'][:int(sample['seq_len'])].numpy().astype(np.float32,copy=True)))
        audit=dict(task=task,subject=subject,source_sentences=n_total,retained=len(rows),rejected=dict(rejected))
        del contents,sentences; gc.collect()
        return rows,audit

    def prepare_data(self):
        rows=[]; audits=[]; sources=[]
        for task in self.cfg['tasks']:
            inventory=osf_files(task)
            chosen=inventory if self.cfg['subjects']=='ALL' else [x for x in inventory if x['subject'] in self.cfg['subjects']]
            if self.cfg['subjects']!='ALL':
                missing=set(self.cfg['subjects'])-{x['subject'] for x in chosen}
                if missing: raise ValueError(f'Missing subjects for {task}: {missing}')
            print(task,len(chosen),'files;',round(sum(x['bytes'] for x in chosen)/1e9,2),'GB before caching')
            for info in chosen:
                cache=self.cache/f"{CACHE_VERSION}-{task}-{info['subject']}.npz"
                if cache.exists():
                    part,audit,old=read_cache(cache)
                    if old.get('sha256')!=info.get('sha256'): raise RuntimeError('Source hash changed; use a new cache version.')
                else:
                    if shutil.disk_usage(self.work).free<info['bytes']+2*1024**3: raise RuntimeError('Not enough disk space.')
                    path=download_checked(info,self.work/'mat'/info['name'])
                    part,audit=self.convert_mat(path,task,info['subject'])
                    write_cache(cache,part,audit,info)
                    path.unlink()  # Delete only the temporary source just downloaded and cached.
                rows.extend(part); audits.append(audit); sources.append(info)
                del part; gc.collect()
        self.set_splits(rows,audits,sources)

    def set_splits(self,rows,audits,sources):
        if self.cfg['split_policy']=='global_sentence_hash':
            for r in rows:
                bucket=stable_seed(r['sentence_key'],self.cfg['seed'])%100
                r['phase']='train' if bucket<80 else 'dev' if bucket<90 else 'test'
        elif self.cfg['split_policy']!='paper_index': raise ValueError('Unknown split policy')
        frame=pd.DataFrame([{k:v for k,v in r.items() if k not in {'labels','eeg'}} for r in rows])
        crossing=frame.groupby('sentence_key').phase.nunique()
        bad=frame[frame.sentence_key.isin(crossing[crossing>1].index)]
        if not bad.empty:
            bad.to_csv(self.persist/'split_conflicts.csv',index=False)
            raise ValueError('Sentence overlap across splits. See split_conflicts.csv. Resolve before training; '
                             'global_sentence_hash is an explicitly different split option.')
        self.splits={}
        for phase in ['train','dev','test']:
            selected=sorted([r for r in rows if r['phase']==phase],key=lambda r:r['id'])
            limit=self.cfg['limits'][phase]
            if limit is not None and len(selected)>limit:
                rng=np.random.default_rng(stable_seed(phase,self.cfg['seed']))
                selected=[selected[i] for i in sorted(rng.choice(len(selected),limit,replace=False))]
            if not selected: raise RuntimeError(f'Empty {phase} set')
            self.splits[phase]=selected
        h=hashlib.sha256()
        for phase,selected in self.splits.items():
            for r in selected:
                h.update((phase+r['id']+r['raw_text']).encode()); h.update(r['labels'].tobytes()); h.update(r['eeg'].tobytes())
        manifest=dict(self.cfg,data_fingerprint=h.hexdigest(),author_commit=A1_COMMIT,
                      model_id=MODEL_ID,model_revision=MODEL_REVISION,feature_cache_version=CACHE_VERSION)
        self.signature=hashlib.sha256(json.dumps(manifest,sort_keys=True).encode()).hexdigest()[:12]
        self.run_dir=self.persist/'runs'/f"{self.cfg['run_name']}_{self.signature}"
        self.report_dir=self.run_dir/'results'; self.report_dir.mkdir(parents=True,exist_ok=True)
        self.split_frame=pd.DataFrame([{k:v for k,v in r.items() if k not in {'eeg','labels'}}
                                     for selected in self.splits.values() for r in selected])
        self.split_frame.to_csv(self.report_dir/'split_manifest.csv',index=False)
        frame.to_csv(self.report_dir/'all_retained_examples.csv',index=False)
        atomic_json(self.report_dir/'run_config.json',manifest)
        atomic_json(self.report_dir/'data_audit.json',dict(sources=sources,filters=audits))
        (self.report_dir/'environment.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
        print(self.split_frame.groupby(['phase','task']).size().unstack(fill_value=0))
        print('No normalized sentence crosses a split. Truncated targets:',int(self.split_frame.target_truncated.sum()))
        print('Run folder:',self.run_dir)

    def dataset(self,phase,condition):
        return PairedDataset(self.splits[phase],condition,self.tokenizer.pad_token_id,self.cfg['noise_seed'])

    def loader(self,phase,condition,shuffle=False,epoch=0,batch_size=None):
        return DataLoader(self.dataset(phase,condition),batch_size=batch_size or self.cfg['micro_batch'],
            shuffle=shuffle,num_workers=0,generator=torch.Generator().manual_seed(self.cfg['seed']+epoch),
            pin_memory=self.device.type=='cuda')

    def paired_input_check(self):
        a=self.dataset('train','EEG')[0]; b=self.dataset('train','noise')[0]
        assert torch.equal(a['mask'],b['mask']) and torch.equal(a['labels'],b['labels'])
        assert not torch.equal(a['x'],b['x']) and torch.isfinite(a['x']).all()
        assert torch.equal(b['x'],self.dataset('train','noise')[0]['x'])
        print('Paired real EEG/noise checks passed. Input shape:',tuple(a['x'].shape))

    def build_model(self):
        seed_all(self.cfg['seed'])
        # The pinned official BART-large revision provides PyTorch .bin weights, not safetensors.
        lm=BartForConditionalGeneration.from_pretrained(MODEL_ID,revision=MODEL_REVISION,
                    use_safetensors=False,attn_implementation='eager')
        model=self.author_models.BrainTranslator(lm,in_feature=840,decoder_embedding_size=lm.config.d_model,
                    additional_encoder_nhead=8,additional_encoder_dim_feedforward=2048)
        if self.cfg['freeze_lm']:
            for p in model.pretrained.parameters(): p.requires_grad_(False)
        if self.cfg['gradient_checkpointing']:
            model.pretrained.gradient_checkpointing_enable(gradient_checkpointing_kwargs={'use_reentrant':False})
        model.pretrained.config.use_cache=False
        return model.to(self.device)

    def amp(self):
        return torch.autocast('cuda',dtype=torch.float16) if self.device.type=='cuda' and self.cfg['amp'] else contextlib.nullcontext()

    def move(self,batch): return {k:v.to(self.device,non_blocking=True) for k,v in batch.items()}

    def forward(self,model,b): return model(b['x'],b['mask'].long(),~b['mask'],b['labels'])

    @torch.no_grad()
    def dev_loss(self,model,condition):
        model.eval(); values=[]
        for batch in self.loader('dev',condition,batch_size=1):
            with self.amp(): values.append(float(self.forward(model,self.move(batch)).loss))
        loss=float(np.mean(values))
        if not np.isfinite(loss): raise FloatingPointError('Nonfinite validation loss; try amp=False in a new run.')
        return loss

    def groups(self,dl):
        group=[]
        for batch in dl:
            group.append(batch)
            if len(group)==self.accum: yield group; group=[]
        if group: yield group

    def train(self,condition,allow_cpu=False):
        if self.device.type!='cuda' and not allow_cpu: raise RuntimeError('Select a GPU runtime before training BART-large.')
        folder=self.run_dir/condition; folder.mkdir(exist_ok=True)
        checkpoint=folder/'best.pt'; complete=folder/'complete.json'
        if complete.exists() and checkpoint.exists():
            assert json.loads(complete.read_text())['signature']==self.signature
            print(condition,'already completed; reusing checkpoint.'); return checkpoint
        model=self.build_model()
        optimizer=torch.optim.SGD([p for p in model.parameters() if p.requires_grad],lr=self.cfg['lr'],momentum=.9)
        scheduler=torch.optim.lr_scheduler.StepLR(optimizer,step_size=30,gamma=.1)
        scaler=torch.amp.GradScaler('cuda',enabled=self.device.type=='cuda' and self.cfg['amp'])
        best,start,history=float('inf'),0,[]
        if checkpoint.exists() and self.cfg['save_resume_state']:
            saved=torch.load(checkpoint,map_location='cpu',weights_only=True)
            assert saved['signature']==self.signature and saved['condition']==condition
            model.load_state_dict(saved['model']); optimizer.load_state_dict(saved['optimizer'])
            scheduler.load_state_dict(saved['scheduler']); scaler.load_state_dict(saved['scaler'])
            best,start,history=saved['best_loss'],saved['epoch']+1,saved['history']
            del saved; print('Resuming from best validation epoch',start)
        total=sum(p.numel() for p in model.parameters()); trained=sum(p.numel() for p in model.parameters() if p.requires_grad)
        print(condition,'parameters:',total,'trainable:',trained)
        if self.device.type=='cuda': torch.cuda.reset_peak_memory_stats()
        begin=time.time()
        for epoch in range(start,self.cfg['epochs']):
            seed_all(self.cfg['seed']+epoch); model.train(); started=time.time(); loss_sum,n_total=0.,0
            dl=self.loader('train',condition,shuffle=True,epoch=epoch)
            for group in tqdm(self.groups(dl),total=math.ceil(len(dl)/self.accum),desc=f'{condition} epoch {epoch+1}'):
                optimizer.zero_grad(set_to_none=True)
                group_tokens=sum(int((b['labels']!=-100).sum()) for b in group)
                for raw in group:
                    b=self.move(raw); n=int((b['labels']!=-100).sum())
                    with self.amp(): loss=self.forward(model,b).loss; weighted_loss=loss*(n/group_tokens)
                    if not torch.isfinite(loss): raise FloatingPointError('Nonfinite training loss; try amp=False.')
                    scaler.scale(weighted_loss).backward(); loss_sum+=float(loss.detach())*n; n_total+=n
                scaler.step(optimizer); scaler.update()
            scheduler.step(); val=self.dev_loss(model,condition)
            entry=dict(epoch=epoch+1,train_token_loss=loss_sum/n_total,dev_sentence_mean_loss=val,
                       seconds=time.time()-started,gpu_peak_GiB=torch.cuda.max_memory_allocated()/2**30 if self.device.type=='cuda' else None)
            history.append(entry); print(entry)
            pd.DataFrame(history).to_csv(self.report_dir/f'training_{condition}.csv',index=False)
            if val<best:
                best=val
                payload=dict(signature=self.signature,condition=condition,epoch=epoch,best_loss=best,
                             model=model.state_dict(),history=history)
                if self.cfg['save_resume_state']:
                    payload.update(optimizer=optimizer.state_dict(),scheduler=scheduler.state_dict(),scaler=scaler.state_dict())
                temp=Path(str(checkpoint)+'.tmp'); torch.save(payload,temp); os.replace(temp,checkpoint); del payload
                print('Saved best checkpoint:',checkpoint)
        if not checkpoint.exists(): raise RuntimeError('No checkpoint saved.')
        atomic_json(complete,dict(signature=self.signature,condition=condition,epochs_completed=self.cfg['epochs'],
            best_dev_loss=best,this_session_seconds=time.time()-begin,parameters=total,trainable_parameters=trained))
        del model,optimizer,scheduler,scaler; gc.collect()
        if self.device.type=='cuda': torch.cuda.empty_cache()
        return checkpoint

    @torch.inference_mode()
    def generate_without_reference(self,model,x,mask,seed):
        # No label, reference text, or reference-dependent output length is accepted here.
        seed_all(seed); encoded=model.addin_forward(x,~mask)
        return model.pretrained.generate(inputs_embeds=encoded,attention_mask=mask.long(),max_length=LENGTH,
            num_beams=5,do_sample=self.cfg['do_sample'],repetition_penalty=5.0,no_repeat_ngram_size=2,use_cache=True)

    @torch.inference_mode()
    def teacher_forced_ids(self,model,b):
        ids=self.forward(model,b).logits[0].argmax(-1).tolist()
        if self.tokenizer.eos_token_id in ids: ids=ids[:ids.index(self.tokenizer.eos_token_id)]
        return ids

    def evaluate(self):
        all_rows=[]
        for train_input in ['EEG','noise']:
            model=self.build_model()
            saved=torch.load(self.run_dir/train_input/'best.pt',map_location='cpu',weights_only=True)
            assert saved['signature']==self.signature and saved['condition']==train_input
            model.load_state_dict(saved['model']); epoch=saved['epoch']+1; del saved
            model.pretrained.gradient_checkpointing_disable(); model.pretrained.config.use_cache=True; model.eval()
            for test_input in ['EEG','noise']:
                rows=[]; ds=self.dataset('test',test_input)
                for i in tqdm(range(len(ds)),desc=f'{train_input} trained / {test_input} test'):
                    b=self.move({k:v.unsqueeze(0) for k,v in ds[i].items()}); r=self.splits['test'][i]
                    started=time.time()
                    ids=self.generate_without_reference(model,b['x'],b['mask'],stable_seed(r['id'],self.cfg['generation_seed']))[0].cpu().tolist()
                    text=self.tokenizer.decode(ids,skip_special_tokens=True)
                    tf_ids=self.teacher_forced_ids(model,b)
                    common={k:r[k] for k in ['id','task','subject','sentence_key','raw_text','reference']}
                    common.update(train_input=train_input,test_input=test_input,checkpoint_epoch=epoch,
                        pair_evaluation_seconds=time.time()-started,
                        reference_tokens=self.tokenizer.convert_ids_to_tokens(r['labels'].tolist(),skip_special_tokens=True))
                    rows.append(dict(common,evaluation='free_generation',prediction=text,prediction_tokens=
                        self.tokenizer.convert_ids_to_tokens(self.tokenizer.encode(text),skip_special_tokens=True)))
                    rows.append(dict(common,evaluation='teacher_forced_diagnostic',
                        prediction=self.tokenizer.decode(tf_ids).replace('<s>',''),prediction_tokens=
                        self.tokenizer.convert_ids_to_tokens(tf_ids,skip_special_tokens=True)))
                atomic_json(self.report_dir/f'predictions_{train_input}_to_{test_input}.json',rows); all_rows.extend(rows)
            del model; gc.collect()
            if self.device.type=='cuda': torch.cuda.empty_cache()
        self.predictions=pd.DataFrame(all_rows)
        self.predictions.drop(columns=['reference_tokens','prediction_tokens']).to_csv(self.report_dir/'predictions.csv',index=False)
        results=[]
        for (a,b,e),group in self.predictions.groupby(['train_input','test_input','evaluation']):
            for task in ['ALL']+sorted(group.task.unique().tolist()):
                subset=group if task=='ALL' else group[group.task==task]
                results.append(dict(train_input=a,test_input=b,evaluation=e,task=task,**score_predictions(subset.to_dict('records'))))
        self.metrics=pd.DataFrame(results); self.metrics.to_csv(self.report_dir/'metrics.csv',index=False)
        return self.metrics

    def report(self):
        import matplotlib.pyplot as plt
        main=self.metrics[(self.metrics.task=='ALL') & (self.metrics.evaluation=='free_generation')].copy()
        free=self.predictions[(self.predictions.train_input=='EEG') & (self.predictions.evaluation=='free_generation')]
        uncertainty=paired_bootstrap(free[free.test_input=='EEG'].to_dict('records'),
            free[free.test_input=='noise'].to_dict('records'),self.cfg['bootstrap_repeats'],self.cfg['seed'])
        uncertainty.to_csv(self.report_dir/'paired_bootstrap.csv',index=False)
        main['condition']=main.train_input+' → '+main.test_input
        fig,ax=plt.subplots(1,2,figsize=(11,4))
        ax[0].bar(main.condition,main.bleu1_bart_pct,color='#2d647d'); ax[0].set_ylabel('BLEU-1 on BART tokens (%) — higher is better')
        ax[1].bar(main.condition,main.wer_pct,color='#b56c37'); ax[1].set_ylabel('Word error rate (%) — lower is better')
        for a in ax: a.tick_params(axis='x',rotation=20); a.set_xlabel('Training input → test input')
        fig.suptitle(self.cfg['preset']+': free generation'); fig.tight_layout()
        fig.savefig(self.report_dir/'eeg_noise_comparison.png',dpi=180,bbox_inches='tight'); plt.show(); plt.close(fig)
        tf=self.metrics[(self.metrics.task=='ALL') & (self.metrics.train_input=='EEG')].pivot(index='test_input',columns='evaluation',values='bleu1_bart_pct')
        tf.to_csv(self.report_dir/'teacher_forcing_comparison.csv')
        examples=self.predictions[self.predictions.train_input=='EEG'].pivot(index=['id','reference'],
            columns=['test_input','evaluation'],values='prediction').sort_index().head(8)
        examples.to_csv(self.report_dir/'example_predictions.csv')
        definitions=[('bleu1_bart_pct','higher','Corpus BLEU-1 on BART subwords, no smoothing'),
            ('bleu2_bart_pct','higher','Cumulative corpus BLEU-2 on BART subwords, no smoothing'),
            ('bleu3_bart_pct','higher','Cumulative corpus BLEU-3 on BART subwords, no smoothing'),
            ('bleu4_bart_pct','higher','Cumulative corpus BLEU-4 on BART subwords, no smoothing'),
            ('rouge1_r_pct','higher','Mean ROUGE-1 recall; empty outputs count as zero'),
            ('rouge1_f_pct','higher','Mean ROUGE-1 F1; empty outputs count as zero'),
            ('rougeL_f_pct','higher','Mean ROUGE-L F1; empty outputs count as zero'),
            ('wer_pct','lower','Corpus word edits / reference word count × 100, case-sensitive; can exceed 100'),
            ('cer_pct','lower','Corpus character edits / reference character count × 100'),
            ('sacrebleu_pct','higher','SacreBLEU corpus score, signature saved in metrics.csv')]
        pd.DataFrame(definitions,columns=['metric','better','definition']).to_csv(self.report_dir/'metric_definitions.csv',index=False)
        adaptations=[
            f"Preset {self.cfg['preset']}, tasks {self.cfg['tasks']}, participants {self.cfg['subjects']}, limits {self.cfg['limits']}, epochs {self.cfg['epochs']}.",
            'Pinned author model and normalization; custom Colab download, training, and evaluation wrapper.',
            f"Seed {self.cfg['seed']}: paper states 42; public training code uses 312.",
            f"do_sample={self.cfg['do_sample']}: public evaluation uses sampling with five beams; paper describes beam search.",
            'Modern dependencies; boolean padding masks; gradient accumulation; optional AMP and gradient checkpointing.',
            'Fixed per-example Gaussian noise; same input lengths and masks retained; random draws differ from original runs.',
            'Sentence-overlap audit; explicit filtering of zero/overlength/nonfinite inputs; selected split policy is recorded.',
            'Free generation does not receive labels; teacher-forced diagnostic scores are separate.',
            'Primary ROUGE includes empty outputs as zero; ignore-empty scores are exported separately.',
            'Paired sentence-group bootstrap is an additional analysis; it does not capture variation between training seeds.',
        ]
        atomic_json(self.report_dir/'adaptations.json',adaptations)
        atomic_json(self.report_dir/'hardware.json',dict(torch=str(torch.__version__),device=str(self.device),
            gpu=torch.cuda.get_device_name(0) if self.device.type=='cuda' else None))
        report=['# EEG versus noise experiment report','',f"Run: {self.run_dir.name}",
            'Status: completed adapted BART experiment. This is not a full replication of every paper result.','',
            '## Data and split',self.split_frame.groupby(['phase','task']).size().to_string(),'',
            '## Free generation',main[['train_input','test_input','n_examples','bleu1_bart_pct','bleu4_bart_pct','rouge1_r_pct','wer_pct']].to_string(index=False),'',
            '## Paired EEG benefit for the EEG-trained model',uncertainty.to_string(index=False),'',
            'Positive differences favor EEG. These are percentage-point performance differences, not percentages of information. '
            'A small or zero gap can reflect limited data/training or model/metric limitations. It does not show that EEG lacks linguistic information.','',
            '## Adaptations and limitations',*['- '+x for x in adaptations],'',
            '## Suggested slide sequence','1. Research question and four input conditions.',
            '2. Model, dataset, split, and metric definitions.','3. Free-generation scores and comparison figure.',
            '4. Teacher-forcing comparison and example sentences.','5. Limitations and measured computing requirements.',
            '6. Next experiment and the proposed LLM-versus-EEG information decomposition.','',
            '## Sources','https://www.nature.com/articles/s41598-025-29587-x',
            'https://github.com/khu-aims/EEG-To-Text/tree/'+A1_COMMIT,
            'https://github.com/MikeWangWZHL/EEG-To-Text','https://osf.io/q3zws/']
        (self.report_dir/'meeting_report.md').write_text('\n'.join(report))
        archive=shutil.make_archive(str(self.work/f'{self.run_dir.name}_results'),'zip',self.report_dir)
        print('Results folder:',self.report_dir,'\nDownload archive:',archive)
        return main,tf,examples,uncertainty,archive

## 4. Fetch the author code and tokenizer
The notebook imports the authors' EEG encoder/model and sample-normalization function from a pinned commit. It suppresses one verbose tensor print. A tokenizer converts text into the word pieces used by BART; model weights are loaded later.

In [ ]:
experiment = EEGExperiment(CFG, WORK, PERSIST)
experiment.setup()

## 5. Download, preprocess, and audit real EEG
Each input contains 840 features per fixated word: 105 electrodes × 8 frequency bands. These are processed, word-aligned EEG features, not raw voltage recordings. SR is movie-review reading; NR is ordinary Wikipedia reading; TSR is task-specific reading.

Downloads are discovered through the public OSF API and checked against the provided SHA-256 hashes. Interrupted downloads resume on rerun. One MATLAB file is decoded at a time, and compact arrays are cached before its temporary source is removed.

The default split uses the first 80% / next 10% / last 10% of sentence positions **before** invalid samples are removed. Identical normalized sentences are checked across every selected task and participant. If any cross splits, execution stops with `split_conflicts.csv`. Resolve this before training; `global_sentence_hash` keeps identical text together but changes the protocol. Pilot limits are applied after this audit.

The development set selects the checkpoint. Test scores never select a checkpoint.

In [ ]:
experiment.prepare_data()
experiment.paired_input_check()
display(experiment.split_frame.groupby(['phase','task']).size().unstack(fill_value=0))

## 6. Train the EEG model
Both training conditions start from the same model initialization. This model learns using real EEG. Training normally uses teacher forcing—the correct previous words help predict the next word. The later **free-generation evaluation** does not do this.

The language model is updated by default. Small GPU batches accumulate gradients toward an effective batch of 32. This reduces memory use without claiming bit-for-bit equivalence to the original run.

The first epoch's measured duration is your runtime estimate. If GPU memory runs out, restart the runtime, keep micro-batch size 1 and gradient checkpointing enabled, then rerun. If storage is tight, disable resume-state saving before creating a new run.

In [ ]:
eeg_checkpoint = experiment.train('EEG')

## 7. Train the noise model
This model receives fixed standard-normal random arrays instead of EEG values. It sees the same targets, split, and input masks. A mask records which word positions exist; both conditions retain that information. Therefore this is a matched random-input control, not a pure language-model-only test.

Random inputs are repeatable for each example and independent of its target words. The generated arrays differ from the authors' particular random draws. The current PyTorch encoder receives boolean padding masks to express the intended masking.

In [ ]:
noise_checkpoint = experiment.train('noise')

## 8. Evaluate all four conditions

| Trained on | Tested on | Purpose |
|---|---|---|
| EEG | EEG | Main decoding condition |
| EEG | Noise | Dependence on EEG values in the EEG-trained model |
| Noise | Noise | Random-input training baseline |
| Noise | EEG | Reverse control |

Each is evaluated with free generation and a separately labeled teacher-forced diagnostic. Free generation receives only EEG-derived embeddings and the input mask—**no reference words, labels, or reference-dependent output length**. Generation seeds are paired across input conditions.

**Metric definitions:** BLEU-1 through BLEU-4 use BART subword tokens, matching the code's token convention. ROUGE measures overlap. WER is word edit distance divided by reference word count and can exceed 100%. These do not directly measure recovered meaning or information in bits. All displayed scores are percentages; higher BLEU/ROUGE is better, lower WER/CER is better. Primary ROUGE counts empty outputs as zero; separate `ignore_empty` columns reproduce that choice in the public evaluator. SacreBLEU and CER are also saved.

In [ ]:
metrics = experiment.evaluate()
display(metrics.loc[metrics.task.eq('ALL'), ['train_input','test_input','evaluation',
    'n_examples','bleu1_bart_pct','bleu4_bart_pct','rouge1_r_pct','wer_pct','empty_predictions']])

## 9. Compare, plot, and write the report
The paired analysis compares EEG versus noise inputs to the **same EEG-trained model**, using free generation. BLEU benefit is EEG minus noise; WER benefit is noise minus EEG. Positive values favor EEG. These are percentage-point score differences, not percentages of information.

The bootstrap resamples sentence groups and keeps repeated recordings of a sentence together. This is an added analysis, not a claim to reproduce the paper's exact uncertainty procedure. It reflects test-sample variation for these fixed models and random draws; it does not capture variability across training seeds or people. The one-person pilot cannot establish broad conclusions about EEG.

Inspect the example sentences as well as scores. Are the outputs generic? Do names, actions, and negations survive? Examples are selected by stable sample order, not by quality.

In [ ]:
main_results, tf_comparison, examples, uncertainty, archive = experiment.report()
display(main_results[['train_input','test_input','n_examples','bleu1_bart_pct','wer_pct']])
display(uncertainty)
display(tf_comparison)
display(examples)

## 10. Download the results
The ZIP contains results, configuration, data/split audits, figures, and a meeting report with a slide outline. It excludes the dataset and large model checkpoints, which stay in your experiment folder.

For your discussion, start with `metrics.csv`, `example_predictions.csv`, `eeg_noise_comparison.png`, `paired_bootstrap.csv`, and `meeting_report.md`. The report records deviations from A.1 so you can describe what you actually reproduced.

In [ ]:
from google.colab import files
files.download(archive)

## What to do after the pilot
For a larger run without downloading additional participants, keep the pilot tasks and participant, change `epochs` to 30 and `limits` to `{'train':None,'dev':None,'test':None}` in `CFG`, then rerun from section 4. This creates a new run and retrains from the same initialization. It is still a one-participant adaptation, not a full paper reproduction.

Then expand participant/sample coverage, repeat with additional model/noise seeds, inspect A.2's evaluation criteria, and consider GLIM. The information decomposition remains a later research step; an EEG/noise performance gap is not that decomposition by itself.

This notebook does not implement T5, Pegasus, ZuCo 2.0, GLIM, or all robustness controls. It establishes a concrete first experiment for the week's reproduction work.

If a cell fails, share its complete error and the GPU name printed in section 4. Data-download failures can be retried in section 5; verified compact caches are reused. Checkpoint recovery is governed by the setting in section 2.

**Describe only completed work:** after the training/evaluation cells finish, you can say you ran an adapted BART pilot with real EEG, tested all four input combinations, and separated free generation from teacher-forced scoring. Before that, say you prepared the implementation.

## Validation before delivery
The notebook format and every code cell's syntax were checked. Both default EEG files (ZAB SR and ZAB NR) were downloaded and preprocessed successfully. Tests used a small, randomly initialized BART with the authors' full EEG encoder to exercise EEG/noise training, saved checkpoint recovery, generation without references, all eight evaluation conditions, known-answer metric checks, paired bootstrap calculations, and report/ZIP exports.

These are software checks, not research results. Full pretrained BART-large training and GPU execution have **not** been completed here; you will run them in Colab. The notebook contains no prefilled experimental scores.